In [2]:
import pandas as pd
import numpy as np
from factors_engineering import *
import gc
df = pd.read_parquet('hs300_data.parquet')
df

,TradingDate,Symbol,OpenPrice,ClosePrice,HighPrice,LowPrice,Volume,Amount,ChangeRatio,TotalShare,...,F060201C,F060301C,F060401C,PE,PB,PS,Year,AvgPrice,Simple_Ret,Log_Ret
0,2010-01-05,000001,1155.064,1133.178,1162.359,1106.429,55649982.0,1.293477e+09,-0.01729,3105433762,...,1.041037,-1.123885,-16.010482,14.382927,3.534831,5.062075,2010,23.243079,-0.017292,-0.017444
1,2010-01-05,000002,914.402,901.352,915.272,887.431,184862078.0,1.910014e+09,-0.02264,10995210218,...,1.086545,0.274957,1.729696,21.372605,2.508569,2.330360,2010,10.332105,-0.022641,-0.022901
2,2010-01-05,000009,47.647,46.204,47.910,45.766,31463320.0,3.320040e+08,-0.02583,1090750529,...,1.302411,0.306810,2.279697,45.444410,3.858595,3.523082,2010,10.552095,-0.025828,-0.026167
3,2010-01-05,000012,202.470,199.852,204.251,197.966,10199239.0,1.944016e+08,-0.01700,1223738124,...,0.989766,0.308631,2.877369,28.065486,4.161223,4.422898,2010,19.060405,-0.016999,-0.017145
4,2010-01-05,000021,116.446,119.477,122.152,115.465,20586249.0,2.766909e+08,0.02290,879518521,...,1.145383,0.031380,1.453359,45.570924,2.981567,0.879170,2010,13.440569,0.022902,0.022644
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2351524,2024-12-31,688472,12.308,12.723,13.047,12.267,44994177.0,5.635011e+08,0.03120,3688217324,...,1.459607,0.110176,2.094439,15.955231,2.156634,0.902834,2024,12.523868,0.031204,0.030727
2351525,2024-12-31,688506,192.970,191.730,197.400,191.010,750140.0,1.454740e+08,-0.00509,401000000,...,1.053099,0.745581,1.059434,27.066889,506.235987,136.835264,2024,193.929183,-0.005085,-0.005098
2351526,2024-12-31,688561,28.560,26.830,28.690,26.830,6358241.0,1.744505e+08,-0.05060,685172377,...,0.932970,-0.058607,-4.523994,256.209924,1.805473,2.853428,2024,27.436913,-0.050602,-0.051927
2351527,2024-12-31,688599,21.392,20.561,21.413,20.561,18658830.0,3.669588e+08,-0.03596,2179365328,...,0.981195,0.184174,7.330657,7.604313,1.150976,0.370942,2024,19.666765,-0.035962,-0.036625


In [2]:
df = calculate_all_factors(df)

开始计算因子...
正在计算：动量趋势类因子 (优化版：最大窗口 20d)...
正在计算：估值市值类因子 (优化版：最大窗口 20d)...
正在计算：质量营运类因子 (优化版：最大窗口 20d)...
正在计算：风险波动类因子 (最大窗口 20d)...
正在计算：成交情绪类因子 (优化版：最大窗口 20d)...


c:\Users\HP\Desktop\FYP\factors_engineering.py:349: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['factor_turnover_chg'] = df['TurnoverRate1'].shift(1) / (df['TurnoverRate1'].shift(6).fillna(method='ffill') + 1e-9)


正在生成预测标签 (Next-day Log Return)...
所有因子计算完成！当前因子维度: 84


In [3]:
df[(df['TradingDate'] == '2010-02-04') & (df['Symbol'] == '000001')]['Log_Ret']

10310   -0.016972
Name: Log_Ret, dtype: float64

In [5]:
# 3. 筛选并提取关键数据列
# 这样做可以丢弃掉中间变量（如 ma_60, ema_12 等），只保留学术论文需要的核心矩阵
keys = ['TradingDate', 'Symbol']
labels = ['target_log_ret']
# 动态获取所有以 factor_ 开头的列
features = [col for col in df.columns if col.startswith('factor_')]

In [6]:
# 创建最终的精简数据集
df_final = df[keys + features + labels].copy()

In [7]:
# 4. 数据清理（关键！）
# 计算过程中的 rolling(250) 等会在开头产生空值，shift(-1) 会在最后一天产生空值
initial_rows = len(df_final)
df_final = df_final.dropna()
print(f"清理完成。剔除初始/末尾空值行数: {initial_rows - len(df_final)}")

清理完成。剔除初始/末尾空值行数: 17160


In [8]:
df_final

,TradingDate,Symbol,factor_mom_2d,factor_mom_5d,factor_mom_10d,factor_mom_20d,factor_reversal_5d,factor_pos_10d,factor_pos_20d,factor_bias_10d,...,factor_turnover_chg,factor_illiquidity,factor_br,factor_avg_px_bias,factor_up_vol_ratio,factor_daily_pos,factor_pvt_20d,factor_turnover_bias,factor_pv_divergence,target_log_ret
21,2010-02-03,000001,-0.015207,-0.030839,-0.038687,-0.082833,0.030839,0.151023,0.249998,-0.025758,...,0.830036,0.000986,0.801502,-0.979179,0.416757,0.157150,-3.173486e+06,0.618633,0,-0.016972
22,2010-02-04,000001,0.063089,0.030580,0.056128,-0.013974,-0.030580,0.644905,0.642860,0.023805,...,1.369481,0.004663,0.955794,-0.979859,0.469072,1.000000,6.426087e+05,1.167729,0,-0.009050
23,2010-02-05,000001,0.038840,0.020690,-0.021164,-0.019867,-0.020690,0.489803,0.519484,0.008770,...,1.157583,0.002492,0.934634,-0.979382,0.471680,0.431843,5.200428e+05,0.648868,0,-0.020203
24,2010-02-08,000001,-0.025687,0.013826,-0.046794,-0.026548,-0.013826,0.476197,0.454547,0.004612,...,1.147097,0.001498,0.893853,-0.979644,0.472304,0.648357,3.349373e+05,0.591208,0,0.020203
25,2010-02-09,000001,-0.028829,0.015066,-0.028391,-0.046017,-0.015066,0.350010,0.336844,-0.012639,...,0.729167,0.004473,0.783953,-0.979342,0.484571,0.174614,-7.790730e+04,0.453724,0,0.012646
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2351523,2024-12-24,688981,0.113998,0.167853,0.096796,0.104689,-0.167853,0.776543,0.776543,0.097847,...,3.060301,0.000068,1.909707,0.013404,0.651586,0.282143,2.113348e+07,2.156261,0,0.011599
2351524,2024-12-25,688981,0.027365,0.178489,0.100682,0.138190,-0.178489,0.869928,0.869928,0.105922,...,2.243137,0.000203,1.897180,-0.013648,0.690539,0.889655,2.317838e+07,1.296538,0,-0.012942
2351525,2024-12-26,688981,0.029523,0.174940,0.128657,0.113523,-0.174940,0.902636,0.902636,0.104735,...,2.034596,0.000121,1.963393,0.002524,0.695111,0.470760,2.168925e+07,1.379041,0,0.008031
2351526,2024-12-27,688981,-0.001342,0.132140,0.109671,0.110818,-0.132140,0.834857,0.834857,0.078902,...,1.055621,0.000186,1.866950,0.005330,0.690281,0.212454,2.141384e+07,0.989078,0,0.018090


In [9]:
# 5. 转换精度 (从 float64 转为 float32)，模型训练更快且节省 50% 硬盘空间
df_final[features + labels] = df_final[features + labels].astype('float32')

In [10]:
# 6. 保存为最终版特征文件
df_final.to_parquet('final_feature_matrix.parquet', index=False)

print("--- 任务成功完成 ---")
print(f"最终特征矩阵形状: {df_final.shape}")
print(f"数据保存路径: final_feature_matrix.parquet")

--- 任务成功完成 ---
最终特征矩阵形状: (2334369, 87)
数据保存路径: final_feature_matrix.parquet


In [11]:
# 手动释放内存
del df, df_final
gc.collect()

0